In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("aws_s3.ventas.bronze_ventas_raw")

print("Registros originales:", df.count())
display(df.limit(5))

In [0]:
# Limpiar espacios y convertir cadenas vacías en NULL
columnas_texto = [
    "id_venta", "id_cliente", "fecha_venta", "producto",
    "cantidad", "precio_unitario_crc", "descuento_pct",
    "total_crc", "metodo_pago", "canal_venta", "provincia"
]

for columna in columnas_texto:
    df = df.withColumn(
        columna,
        F.when(
            F.trim(F.col(columna).cast("string")) == "",
            None
        ).otherwise(F.trim(F.col(columna).cast("string")))
    )

# Estandarizar textos categóricos
for columna in ["producto", "metodo_pago", "canal_venta", "provincia"]:
    df = df.withColumn(
        columna,
        F.initcap(F.lower(F.col(columna)))
    )

# Normalizar y convertir fechas (try_to_date devuelve NULL ante fechas imposibles como 31/02)
df = df.withColumn(
    "fecha_normalizada",
    F.coalesce(
        F.try_to_date("fecha_venta", "yyyy-MM-dd"),
        F.try_to_date("fecha_venta", "dd/MM/yyyy")
    )
)

# Convertir cantidades y descuentos
df = df.withColumn(
    "cantidad_limpia",
    F.col("cantidad").cast("decimal(18,2)")
)

df = df.withColumn(
    "descuento_limpio",
    F.col("descuento_pct").cast("decimal(8,2)")
)

# Limpiar símbolo de moneda y separadores de miles
for columna in ["precio_unitario_crc", "total_crc"]:
    df = df.withColumn(
        columna + "_limpio",
        F.regexp_replace(F.col(columna), "[₡,\\s]", "")
         .cast("decimal(18,2)")
    )

In [0]:
'''Detectar duplicados y validar los registros
Para evitar eliminar información sin dejar evidencia, clasificaremos las ventas duplicadas utilizando id_venta como clave de negocio. Conservaremos una sola aparición de cada ID, dando prioridad al registro con la fecha de ingesta más antigua cuando esté disponible. '''

# Orden para identificar la primera aparición de cada venta
ventana = Window.partitionBy("id_venta").orderBy(
    F.col("ingestion_timestamp").asc_nulls_last(),
    F.col("source_file").asc_nulls_last()
)

df = df.withColumn(
    "numero_duplicado",
    F.row_number().over(ventana)
)

# Calcular el total esperado
df = df.withColumn(
    "total_esperado",
    F.round(
        F.col("cantidad_limpia") *
        F.col("precio_unitario_crc_limpio") *
        (1 - F.col("descuento_limpio") / 100),
        2
    )
)

# Detectar diferencias entre el total original y el esperado
df = df.withColumn(
    "total_revisar",
    F.when(
        F.col("total_crc_limpio").isNull() |
        (F.abs(
            F.col("total_crc_limpio") - F.col("total_esperado")
        ) > 0.01),
        True
    ).otherwise(False)
)

# Construir la lista de errores críticos
df = df.withColumn(
    "errores_calidad",
    F.concat_ws(
        ", ",
        F.when(F.col("id_venta").isNull(), "id_venta_vacio"),
        F.when(F.col("id_cliente").isNull(), "id_cliente_vacio"),
        F.when(F.col("fecha_normalizada").isNull(), "fecha_invalida"),
        F.when(
            F.col("cantidad_limpia").isNull() |
            (F.col("cantidad_limpia") <= 0),
            "cantidad_invalida"
        ),
        F.when(
            F.col("precio_unitario_crc_limpio").isNull() |
            (F.col("precio_unitario_crc_limpio") < 0),
            "precio_invalido"
        ),
        F.when(
            F.col("descuento_limpio").isNull() |
            (F.col("descuento_limpio") < 0) |
            (F.col("descuento_limpio") > 100),
            "descuento_invalido"
        ),
        F.when(
            F.col("numero_duplicado") > 1,
            "venta_duplicada"
        )
    )
)

In [0]:
# Registros con errores críticos
df_rechazadas = df.filter(
    F.col("errores_calidad") != ""
)

# Registros válidos
df_validas = df.filter(
    F.col("errores_calidad") == ""
)

# Seleccionar columnas limpias para Silver
columnas_silver = [
    "id_venta",
    "id_cliente",
    "fecha_normalizada",
    "producto",
    "cantidad_limpia",
    "precio_unitario_crc_limpio",
    "descuento_limpio",
    "total_crc_limpio",
    "total_esperado",
    "total_revisar",
    "metodo_pago",
    "canal_venta",
    "provincia",
    "source_file",
    "ingestion_timestamp"
]

df_validas = df_validas.select(*columnas_silver)

print("Ventas válidas:", df_validas.count())
print("Registros rechazados:", df_rechazadas.count())
print("Totales por revisar:", df.filter("total_revisar").count())

In [0]:
# Guardar ventas válidas
(
    df_validas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("aws_s3.ventas.silver_ventas")
)

# Guardar registros rechazados para auditoría
(
    df_rechazadas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("aws_s3.ventas.silver_ventas_rechazadas")
)

print("Capa Silver creada correctamente.")